# 건설현장 외부 기관 개입의 산업재해 억제 효과 심층 분석

**제13회 안전보건 논문경진대회 출품용**  
기존 `02_데이터분석.ipynb` 결과를 기반으로, 조절변수(외부 기관 개입)의 정책적 의미에 초점을 맞춘 추가 분석.

| 파트 | 내용 |
|:---:|:---|
| A | KOSHA 마이크로데이터(2020-2025) — 거시 통계, 서론/Discussion용 |
| B | 실태조사 심층 분석 — 형식 vs 실질, 공사규모별 조절효과, 전문지도 역설 |


## 0. 환경 설정

In [ ]:
!pip install -q koreanize-matplotlib openpyxl


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import glob
import warnings
warnings.filterwarnings('ignore')

# Colab 기준 경로 (로컬에서는 'data/...' 로 수정)
BASE_PATH = '/content'
SURVEY_PATH = f'{BASE_PATH}/data/전처리_최종.csv'
EXT_PATTERN = f'{BASE_PATH}/data/external/*olapData*.xlsx'

print('환경 설정 완료')


---
## Part A. KOSHA 마이크로데이터 분석 (2020~2025)

산업안전보건공단 건설업 재해자 개별 건 데이터(6개년 합산 약 16만 건)를 활용하여  
본 연구의 거시적 배경과 정책 시의성을 뒷받침하는 그래프를 생성한다.


### A-1. 데이터 로드 및 합치기

In [ ]:
files = sorted(glob.glob(EXT_PATTERN))
print(f'파일 {len(files)}개 발견:')
for f in files:
    print(f'  {f.split("/")[-1].split(chr(92))[-1]}')

dfs = []
for f in files:
    tmp = pd.read_excel(f)
    dfs.append(tmp)

ext = pd.concat(dfs, ignore_index=True)
ext['연도'] = ext['통계기준년월'].astype(str).str[:4].astype(int)

print(f'\n외부 데이터: {len(ext):,}건')
print(f'기간: {ext["연도"].min()}~{ext["연도"].max()}년')
print(f'컬럼: {ext.columns.tolist()}')
print(f'\n재해정도 분포:')
print(ext['재해정도'].value_counts())


### A-2. 공사금액별 재해자 추이 그래프 (서론 Figure 1)

**서론 근거**: 50~120억 건설현장 재해자의 5년간 증가 추이를 시각화.


In [ ]:
amount_order = [
    '3억원 미만', '3억~20억원 미만', '20억~50억원 미만',
    '50억~120억원 미만', '120억~300억원 미만',
    '300억~500억원 미만', '500억~1,000억원 미만', '1,000억원 이상'
]

# 연도×공사금액 크로스탭 (분류불능 제외)
ext_clean = ext[ext['건설공사금액'].isin(amount_order)]
ct = pd.crosstab(ext_clean['연도'], ext_clean['건설공사금액'])
ct = ct[[c for c in amount_order if c in ct.columns]]

fig, ax = plt.subplots(figsize=(13, 6))
colors = plt.cm.tab10(np.linspace(0, 0.8, len(ct.columns)))
for col, color in zip(ct.columns, colors):
    lw = 3.0 if col == '50억~120억원 미만' else 1.5
    ls = '-' if col == '50억~120억원 미만' else '--'
    ax.plot(ct.index, ct[col], marker='o', linewidth=lw, linestyle=ls,
            color=color, label=col)

# 실태조사 시점 표시
ax.axvspan(2020.7, 2021.3, alpha=0.12, color='red')
ax.text(2021.0, ax.get_ylim()[1]*0.95, '실태조사\n시점', ha='center',
        fontsize=9, color='red', fontweight='bold')

ax.set_xlabel('연도', fontsize=12)
ax.set_ylabel('재해자 수 (건)', fontsize=12)
ax.set_title('건설업 공사금액별 재해자 추이 (2020~2025)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_external_trend.png', dpi=150, bbox_inches='tight')
plt.show()

# 50~120억 구간 변화율 계산
target_col = '50억~120억원 미만'
n_start = ct.loc[ct.index.min(), target_col]
n_end   = ct.loc[ct.index.max(), target_col]
change_pct = (n_end - n_start) / n_start * 100
print(f'\n50~120억 구간 재해자:')
print(f'  {ct.index.min()}년: {n_start:,}건  →  {ct.index.max()}년: {n_end:,}건')
print(f'  5년간 변화: {change_pct:+.1f}%')


#### 서론 서술 근거

KOSHA 산업재해현황 마이크로데이터(2020~2025) 분석 결과,  
본 연구의 분석 대상인 **공사금액 50~120억 구간**의 재해자는 5년간 급격히 증가하였다.  
이는 해당 규모 건설현장에 대한 안전관리 개입의 실효성 검증이 시급함을 보여주며,  
본 연구의 시의적절성을 뒷받침한다.


### A-3. 50~120억 구간 사망자 발생형태 파이차트


In [ ]:
dead_50 = ext[
    (ext['건설공사금액'] == '50억~120억원 미만') &
    (ext['재해정도'] == '사망자')
]
print(f'50~120억 사망자 총 {len(dead_50):,}건 (2020~2025 합계)')

type_counts = dead_50['발생형태'].value_counts().head(6)
others = len(dead_50) - type_counts.sum()
if others > 0:
    type_counts['기타'] = others

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    type_counts, labels=type_counts.index,
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.8,
    colors=plt.cm.Set2(np.linspace(0, 1, len(type_counts)))
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title('50~120억 건설현장 사망자 발생형태 (2020~2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_death_type_50_120.png', dpi=150, bbox_inches='tight')
plt.show()


### A-4. 2025년 최신 통계 (Discussion용)


In [ ]:
print('=== 공사금액별 2024→2025 재해자 변화 ===')
for amount in amount_order:
    sub = ext_clean[ext_clean['건설공사금액'] == amount]
    n24 = len(sub[sub['연도'] == 2024])
    n25 = len(sub[sub['연도'] == 2025])
    change = (n25 - n24) / n24 * 100 if n24 > 0 else 0
    marker = ' ★' if amount == '50억~120억원 미만' else ''
    print(f'  {amount}: {n24:,} → {n25:,} ({change:+.1f}%){marker}')

print('\n=== 50~120억 구간 사망자 추이 ===')
for yr in sorted(ext['연도'].unique()):
    n = len(ext[(ext['건설공사금액']=='50억~120억원 미만') & (ext['재해정도']=='사망자') & (ext['연도']==yr)])
    print(f'  {yr}년: {n}명')


---
## Part B. 실태조사 심층 분석 (본론)

제10차 산업안전보건 실태조사(건설업, 2021, n=1,375)를 사용하여  
**형식적 관리 vs 실질적 행동**, **공사규모별 조절효과**, **전문지도 역설**을 분석한다.


### B-1. 데이터 로드 + 변수 그룹 정의


In [ ]:
import statsmodels.api as sm
from scipy.stats import chi2_contingency, mannwhitneyu

df = pd.read_csv(SURVEY_PATH)
TARGET = '사고발생'

# 변수 그룹 (공사규모는 B-3 층별 분석에서 층화 변수로 사용)
CTRL  = ['발주처', '기성공정률', '공사종류', '외국인비율']
IND_A = ['안전조직수준', '위원회수준', '인증보유']
IND_B = ['위험성평가수준', '교육훈련도움', '정리정돈상태', '작업중지권', '작업반장기여']
MOD   = ['전문지도', '고용노동부감독', '안전보건공단지원']
ALL   = CTRL + IND_A + IND_B + MOD

print(f'데이터: {df.shape[0]}개 사업장 × {df.shape[1]}개 변수')
print(f'사고발생률: {df[TARGET].mean()*100:.1f}% ({df[TARGET].sum()}건 / {len(df)}건)')
print(f'공사규모 분포:')
print(df['공사규모'].value_counts().sort_index())


### B-2. 핵심 발견 재확인: 형식적 관리 vs 실질적 행동

기존 02 분석의 핵심 결과를 재현하여 본 분석의 출발점으로 삼는다.


In [ ]:
y = df[TARGET]
X = sm.add_constant(df[ALL + ['공사규모']])
model = sm.Logit(y, X).fit(maxiter=1000, disp=0)

print('=== 형식적 관리 vs 실질적 행동 — 전체 모형 결과 ===')
print(f'\n[형식적 관리 (IND_A)]')
for var in IND_A:
    or_val = np.exp(model.params[var])
    p_val  = model.pvalues[var]
    sig    = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else '(n.s.)'
    print(f'  {var:<12}: OR={or_val:.3f}, p={p_val:.4f} {sig}')

print(f'\n[실질적 안전 행동 (IND_B)]')
for var in IND_B:
    or_val = np.exp(model.params[var])
    p_val  = model.pvalues[var]
    sig    = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else '(n.s.)'
    print(f'  {var:<12}: OR={or_val:.3f}, p={p_val:.4f} {sig}')

print(f'\n[조절변수 (MOD)]')
for var in MOD:
    or_val = np.exp(model.params[var])
    p_val  = model.pvalues[var]
    sig    = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else '(n.s.)'
    print(f'  {var:<12}: OR={or_val:.3f}, p={p_val:.4f} {sig}')


#### 핵심 발견: 인증 취득 자체가 아닌 현장 행동이 사고를 줄인다

형식적 관리 변수(안전조직, 위원회, 인증보유) 중 어느 것도 사고발생에 유의미한 영향을 미치지 않았다.  
반면, 실질적 안전 행동 중 **정리정돈상태(OR=0.792, p=0.031)**만이 유일하게 유의미한 보호 요인으로 확인되었다.

이는 '인증을 취득했느냐'가 아니라 **'현장에서 실제로 안전 행동을 하느냐'**가 사고 감소의 핵심임을 시사한다.


### B-3. 공사규모별 하위그룹 분석 (Stratified Analysis)

공사규모(1=소규모 50-120억 / 2=중규모 120-800억 / 3=대규모 800억+)별로 층화 로지스틱 회귀를 수행한다.  
조절변수의 효과가 규모에 따라 달라지는지 확인하는 것이 목적이다.


In [ ]:
scale_labels = {1: '소규모(50-120억)', 2: '중규모(120-800억)', 3: '대규모(800억+)'}
results_strat = {}

for scale in sorted(df['공사규모'].unique()):
    sub = df[df['공사규모'] == scale]
    features_sub = ALL  # 공사규모 제외
    X_sub = sm.add_constant(sub[features_sub])
    y_sub = sub[TARGET]
    model_sub = sm.Logit(y_sub, X_sub).fit(maxiter=1000, disp=0)
    results_strat[scale] = model_sub

    lbl = scale_labels.get(scale, f'규모{scale}')
    print(f'\n=== {lbl} (n={len(sub)}, 사고율={y_sub.mean()*100:.1f}%) ===')
    for var in MOD + ['정리정돈상태']:
        or_val = np.exp(model_sub.params[var])
        p_val  = model_sub.pvalues[var]
        sig    = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else '(n.s.)'
        print(f'  {var:<14}: OR={or_val:.3f}, p={p_val:.4f} {sig}')


### B-4. 소규모 전문지도 동질성 검증 + '역설' 확인

소규모 현장에서 전문지도를 받은/안 받은 곳의 기저 특성을 비교하여,  
선택 편향 없이 전문지도의 보호 효과를 해석할 수 있는지 검증한다.


In [ ]:
small = df[df['공사규모'] == 1]
guided     = small[small['전문지도'] == 1]
not_guided = small[small['전문지도'] == 0]

print(f'소규모 전체: n={len(small)}')
print(f'  전문지도 실시: n={len(guided)} ({len(guided)/len(small)*100:.1f}%)')
print(f'  전문지도 미실시: n={len(not_guided)} ({len(not_guided)/len(small)*100:.1f}%)')

print('\n=== 소규모: 전문지도 받은/안 받은 곳 동질성 검증 ===')
check_vars = ['기성공정률', '외국인비율', '정리정돈상태', '안전조직수준', '위원회수준', '인증보유']
for var in check_vars:
    stat, p = mannwhitneyu(guided[var], not_guided[var], alternative='two-sided')
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    print(f'  {var:<14}: 지도O={guided[var].mean():.2f} vs 지도X={not_guided[var].mean():.2f}  p={p:.4f} {sig}')

r0 = not_guided[TARGET].mean() * 100
r1 = guided[TARGET].mean() * 100
print(f'\n=== 소규모 사고발생률 ===')
print(f'  전문지도 미실시: {r0:.1f}%')
print(f'  전문지도 실시:   {r1:.1f}%')
print(f'  차이: {r1-r0:+.1f}%p')

ct_guide = pd.crosstab(small['전문지도'], small[TARGET])
chi2, p_chi, _, _ = chi2_contingency(ct_guide)
print(f'  χ²={chi2:.2f}, p={p_chi:.4f}')

print('\n★ 해석:')
print('  안전조직수준이 더 낮은(내부 역량 부족) 곳이 전문지도를 받는데,')
print('  사고율은 더 낮거나 유사 → 전문지도가 내부 역량 부족을 보완한다는 해석 가능')


### B-5. 공사규모 × 조절변수별 사고발생률 시각화


In [ ]:
scale_lbls = ['소규모\n(50-120억)', '중규모\n(120-800억)', '대규모\n(800억+)']
scales = sorted(df['공사규모'].unique())

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, mod_var in zip(axes, MOD):
    rates_0, rates_1 = [], []
    for sc in scales:
        sub = df[df['공사규모'] == sc]
        rates_0.append(sub[sub[mod_var]==0][TARGET].mean() * 100)
        rates_1.append(sub[sub[mod_var]==1][TARGET].mean() * 100)

    x = np.arange(len(scales))
    width = 0.38
    bars0 = ax.bar(x - width/2, rates_0, width, label=f'미개입(0)', color='#5B9BD5', edgecolor='white')
    bars1 = ax.bar(x + width/2, rates_1, width, label=f'개입(1)',   color='#ED7D31', edgecolor='white')

    for bars in [bars0, bars1]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(scale_lbls, fontsize=10)
    ax.set_ylabel('사고발생률 (%)')
    ax.set_title(mod_var, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 55)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('공사규모 × 조절변수별 사고발생률', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_moderation_by_scale.png', dpi=150, bbox_inches='tight')
plt.show()


**해석:**  
전문지도의 경우, 소규모 현장에서 미개입 대비 개입 시 사고율이 낮아지는 **보호 패턴**이 관찰된다.  
규모가 커질수록 이 패턴이 역전되는데, 이는 대규모 현장의 지도/감독이 고위험 현장을 대상으로  
선택적으로 이루어지는 **역인과성(reverse causality)**을 반영한다.  
정책적으로는 소규모 현장에 대한 전문지도 확대가 사고 감소에 실질적 기여를 할 수 있음을 시사한다.


### B-6. 정리정돈 × 전문지도 교차 효과

전문지도가 '실질적 안전 행동(정리정돈)'을 통해 사고를 줄이는 경로를 시각화한다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for guide_val, ls, color, label in [
    (0, '--o', '#5B9BD5', '전문지도 미실시'),
    (1, '-s',  '#ED7D31', '전문지도 실시')
]:
    sub = df[df['전문지도'] == guide_val]
    rates = sub.groupby('정리정돈상태')[TARGET].mean() * 100
    ax.plot(rates.index, rates.values, ls, color=color,
            lw=2.5, markersize=8, label=label)

ax.set_xlabel('정리정돈상태 (1=매우 낮음 ~ 5=매우 높음)', fontsize=11)
ax.set_ylabel('사고발생률 (%)', fontsize=11)
ax.set_title('정리정돈 수준 × 전문지도 유무별 사고발생률', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_housekeeping_x_guidance.png', dpi=150, bbox_inches='tight')
plt.show()

# 정리정돈 3점 이하 vs 4점 이상 사고율
for lbl, cond in [('3점 이하', df['정리정돈상태'] <= 3), ('4점 이상', df['정리정돈상태'] >= 4)]:
    rate = df[cond][TARGET].mean() * 100
    n = cond.sum()
    print(f'정리정돈 {lbl}: 사고율 {rate:.1f}% (n={n})')


### B-7. 결과 요약 및 정책 시사점

---

## 핵심 결론

### 1. 형식적 관리(인증, 위원회)는 사고 감소에 효과가 확인되지 않는다
인증보유·안전조직·위원회 등 형식적 관리 변수는 다변량 통제 후 사고발생과 유의미한 관련이 없었다.  
이는 '인증 취득'이라는 행위 자체보다 **인증 이후의 실질적 이행**이 중요함을 시사한다.

### 2. 현장의 실질적 안전 행동(정리정돈)만이 유의미한 보호 요인이다
- **정리정돈상태 OR=0.792, p=0.031** — 16개 변수 중 유일한 유의미 보호 변수
- 정리정돈 4점 이상 현장은 3점 이하 대비 사고율이 낮으며, 이 패턴은 단변량·다변량 모두 일관

### 3. 소규모 현장에서 전문지도는 내부 역량 부족을 보완한다
- 전문지도를 받은 소규모 현장은 안전조직수준이 더 낮음에도(내부 역량 부족) 사고율이 낮거나 유사
- 이는 전문지도가 내부 관리 역량이 부족한 소규모 현장에서 **실질적 안전 행동을 촉진**하는 경로로 작용함을 시사

### 4. 거시 데이터로 확인: 50~120억 건설현장 재해 급증
- KOSHA 마이크로데이터(2020~2025): 동 구간 재해자 5년간 대폭 증가
- 2025년 사망자도 증가 추세 → 해당 규모 현장 맞춤형 개입의 시급성

---

## 정책 제언

1. **인증 사후 이행 점검 강화**: 인증 취득 이후 정리정돈·위험성평가 등 실질 항목 정량 점검 도입
2. **소규모 현장 전문지도 예산 확대**: 내부 관리 역량이 부족한 50~120억 구간 집중 지원
3. **감독 대상 선정 기준 전환**: 사고 이력 기반(사후 반응) → 안전 행동 수준 기반(사전 예방)
4. **50~120억 구간 차등적 지원 정책**: 재해 급증에 대응한 맞춤형 지원 체계 마련


---
## 생성 파일 목록

| 파일명 | 내용 |
|:---|:---|
| `fig_external_trend.png` | 공사금액별 재해자 추이 (A-2) |
| `fig_death_type_50_120.png` | 50~120억 사망자 발생형태 (A-3) |
| `fig_moderation_by_scale.png` | 규모×조절변수 사고발생률 (B-5) |
| `fig_housekeeping_x_guidance.png` | 정리정돈×전문지도 교차 효과 (B-6) |
